# 03. Offline Flash Storage Download & Session Reconstruction

Nuanic rings record physiological metrics internally to NOR flash memory when disconnected from Bluetooth.
This tutorial demonstrates how to:
1. Query flash storage memory capacity and utilization (`read_storage_usage`).
2. Download raw fixed-size binary records from buffer register `7c3b82e7` (`download_storage`).
3. Decode records according to active storage format (Format 1: Raw EDA `<HQI` vs Format 2: DNE `<HQiii`).
4. Reconstruct absolute timestamps using boot counters and millisecond offsets.

In [ ]:
import asyncio
import pandas as pd
from datetime import datetime
from nuanic_ring.core import NuanicConnector

## 1. Connect and Check Memory Usage

In [ ]:
connector = NuanicConnector()
ok = await connector.connect()
if not ok:
    print("Could not connect to ring.")
else:
    usage = await connector.read_storage_usage()
    fmt = await connector.read_storage_format()
    print(f"Storage Format: {fmt}")
    if usage:
        print(f"Size: {usage['size_bytes']} bytes | Used: {usage['used_bytes']} bytes "
              f"({usage['percent_used']:.1f}%)")

## 2. Download and Parse Offline Records

In [ ]:
print("Downloading recorded records from flash buffer...")
records = await connector.download_storage()
print(f"Downloaded {len(records)} records.")

if records:
    df_records = pd.DataFrame(records)
    print(df_records.head())
    # Save offline export to CSV
    df_records.to_csv("data/offline_flash_export.csv", index=False)
    print("Saved to data/offline_flash_export.csv")

await connector.disconnect()